In [1]:
import random
from pathlib import Path

import pandas as pd
from nltk import word_tokenize
from nltk.corpus import stopwords
from sentence_transformers import SentenceTransformer

C:\Users\Work\Desktop\Python projects\mse-group-project\venv\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [2]:
DIR = (Path("*")).parent / "txt_en"

In [3]:
stop_words = set(stopwords.words("english"))

In [4]:
sentence_transformer = SentenceTransformer('all-MiniLM-L6-v2')

In [5]:
docs = [
    (file.stem, file.read_text(encoding="utf-8"))
    for file in DIR.glob("*.txt")
]

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words="english", lowercase=True)
X = vectorizer.fit_transform([text for _, text in docs])

In [18]:
query = "food and drinks"
y = vectorizer.transform([query])
scores = (X @ y.T).toarray().flatten()

In [19]:
scores

array([0., 0., 0., ..., 0., 0., 0.])

In [6]:
all_ids = []
all_paragraphs = []

for fn, text in docs:
    para = text.split("\n\n")

    if len(para) > 64:
        para = random.sample(para, 64)

    all_paragraphs.extend(para)
    all_ids.extend([fn] * len(para))

In [7]:
embeddings = sentence_transformer.encode(
    all_paragraphs,
    show_progress_bar=True,
    normalize_embeddings=True,
    batch_size=64,
)

Batches:   0%|          | 0/3034 [00:00<?, ?it/s]

In [62]:
query = "food and drinks"

In [54]:
search_terms = [
    word.lower() for word in word_tokenize(query)
    if word.lower() not in stop_words
]

In [55]:
search_terms

['food', 'drinks']

In [56]:
search_embeddings = sentence_transformer.encode(
    search_terms,
    show_progress_bar=True,
    # if we normalize, we can simply do a dot product to get the cosine similarity
    normalize_embeddings=True,
)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [57]:
df = pd.DataFrame(
    embeddings @ search_embeddings.T,
    columns=search_terms,
    index=all_ids
)

In [58]:
(
    df
    .groupby(df.index)
    .apply(lambda x: x.max(axis=0).sum() / len(search_terms))
    .sort_values(ascending=False)
)

003943ee2a0b6ad392688c16d788d835_ENG    0.360831
001579eb8e699142183c6db7c469d67c_ENG    0.245849
00247edbdf1cb5cf73741953b5185d40_ENG    0.167926
00216835324133083ca5723c864ba0ad_ENG    0.166588
0034f0325381edad86171744acd356ef_ENG    0.127558
001c6da156a8feb79c9b2f11f6f2214f_ENG    0.107302
0023f24a43f72252c873381b6d4714bc_ENG    0.069227
002a2802e5c4dd1128100fe1572437aa_ENG    0.053308
001a06739dceb4e48ac2c4d93d6aacf4_ENG    0.053071
00015cf9deb96f8073aac331866a4a19_ENG    0.046343
dtype: float64

In [60]:
with open(DIR / "003943ee2a0b6ad392688c16d788d835_ENG.txt", "r", encoding="utf-8") as f:
    print(f.read())

Lounge - Hotel Lamm Tübingen

Delicious swabian food & bevrages.

Our cuisine includes the most natural variety of swabian and non-regional cuisine. Fresh and carefully selected products are the basics: Fantasy, variety and individual wishes are the ingredients. The result might be hard to describe, but is easily enjoyed. Whether vegetarian, solid swabian or as a gourmet meal – the Lamm leaves nothing to be desired. Volker Theurer , your chef, puts emphasis on carefully selected food and beverages. Our guests should feel at home very simple - Ines Possegger and your friendly service team is attentive and happy to entertain you.

Opening times at the restaurant:

Monday - Saturday from 17:00 clock and by appointment

For your celebrations, festivals and events, we are also happy to be confirmed outside of our regular opening times are available!

Distillery opening times - Specialty Sales "in Brennereistüble":

Monday - Friday from 8:00 - 12:00 clock and from 14:00 - 18:00 clock Saturda

In [63]:
!pip install keybert

   ---------------------------------------- 0.0/240.7 kB ? eta -:--:--
   ---------------------------------------- 240.7/240.7 kB 7.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/87.5 kB ? eta -:--:--
   ---------------------------------------- 87.5/87.5 kB 4.8 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.1.1 -> 24.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from keybert import KeyBERT

kw_extractor = KeyBERT('distilbert-base-nli-mean-tokens')

docs = [
    (file.stem, file.read_text(encoding="utf-8"))
    for file in DIR.glob("*.txt")
]
docs = docs[:10]

kw_extractor.extract_keywords(docs[0][1], keyphrase_ngram_range=(1, 1), stop_words=stop_words)

In [70]:
from nltk import word_tokenize
import re

In [100]:
query = "Where is the best pizza in Tübingen?"

query_terms = [
    token.lower() for token in word_tokenize(query)
    if re.match(r'^[\w-]+$', token) and token.lower() not in stop_words
]
query_terms.append(query)

In [101]:
query_terms

['best', 'pizza', 'tübingen', 'Where is the best pizza in Tübingen?']

In [1]:
from pathlib import Path
import numpy as np
import time

In [3]:
DIR = Path("D://embeddings")

In [4]:
ids = np.load(DIR / "ids.npy")

In [29]:
embeddings = np.load(DIR / "all-mpnet-base-v2.npy")

In [30]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [31]:
sentence_transformer = SentenceTransformer('all-mpnet-base-v2')

In [32]:
time_start = time.time()

query = "Where is the best pizza in Tübingen?"

query_terms = [
    token.lower() for token in word_tokenize(query)
    if re.match(r'^[\w-]+$', token) and token.lower() not in stop_words
]
query_terms.append(query)

query_embeddings = sentence_transformer.encode(
    query_terms,
    show_progress_bar=True,
    normalize_embeddings=False,
)

similarities = cosine_similarity(embeddings, query_embeddings)
df = pd.DataFrame(similarities, index=ids, columns=query_terms)

(
    df
    .groupby(df.index)
    .apply(lambda x: x.max(axis=0).sum() / len(query_terms))
    .sort_values(ascending=False)
)

print(f"Time taken: {time.time() - time_start:.2f}s")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Time taken: 3.31s


In [40]:
embeddings.nbytes / 1e9  # this is the size of the embeddings in GB

0.596465664

In [23]:
unstacked_bit = 194162 * 384 * 32

In [24]:
stacked_bit = 64 * 384 * 12_000 * 32

In [26]:
unstacked_gb = unstacked_bit / 8 / 1e9
unstacked_gb

0.298232832

In [28]:
stacked_gb = stacked_bit / 8 / 1e9
stacked_gb

1.179648

In [41]:
path = Path("D://mse_latest/latest.csv")
path.exists()

True

In [42]:
import pandas as pd

In [56]:
df = pd.read_csv(path)

In [55]:
df_clean = df.drop_duplicates(subset=["url"], keep="first")

In [58]:
df.loc[df.url == 'https://computomics.com/home.html']

,doc_id,url,domain,main_domain,depth,priority,status,created,updated,root,random_sort_key,features_tubingen,features_english
1385,6fd2dc3b94d665b8eee192d7f0c6b05d,https://computomics.com/home.html,computomics.com,computomics.com,1,0,completed,2024-07-14 17:39:49.219371,2024-07-14 17:42:08.048509,1fecc3a96c8cca2c244e40947664895f,0.853634,True,True
21606,e49a296326f252bfcc7b8097b90c10be,https://computomics.com/home.html,computomics.com,computomics.com,2,1,completed,2024-07-14 17:42:07.962492,2024-07-14 17:50:21.352036,1fecc3a96c8cca2c244e40947664895f,0.442262,True,True
21607,cc75accf4b966bffcdb92ce05925db3b,https://computomics.com/home.html,computomics.com,computomics.com,2,1,completed,2024-07-14 17:42:07.966484,2024-07-14 17:44:57.969981,1fecc3a96c8cca2c244e40947664895f,0.111388,True,True
22686,f684276715029ed55d36c1f389d64269,https://computomics.com/home.html,computomics.com,computomics.com,3,1,completed,2024-07-14 17:42:34.728013,2024-07-14 18:04:03.103526,1fecc3a96c8cca2c244e40947664895f,0.015218,True,True
22687,00b48504ea437a97a83fc59badb67e54,https://computomics.com/home.html,computomics.com,computomics.com,3,1,completed,2024-07-14 17:42:34.733029,2024-07-14 18:18:25.410292,1fecc3a96c8cca2c244e40947664895f,0.149379,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
219731,7da57e0e7b9ae36f90823530a351d882,https://computomics.com/home.html,computomics.com,computomics.com,4,1,completed,2024-07-14 22:44:14.888264,2024-07-14 22:49:19.965627,1fecc3a96c8cca2c244e40947664895f,0.000022,True,True
219732,9771f74d3b3f3038ee157f2aeff16d9c,https://computomics.com/home.html,computomics.com,computomics.com,4,1,completed,2024-07-14 22:44:14.921761,2024-07-15 03:11:20.027718,1fecc3a96c8cca2c244e40947664895f,0.985590,True,True
219733,ae30f288ce5f2e32a56ab537c428d0b1,https://computomics.com/home.html,computomics.com,computomics.com,4,1,completed,2024-07-14 22:44:14.950940,2024-07-14 23:25:24.050476,1fecc3a96c8cca2c244e40947664895f,0.202721,True,True
219734,c8fd401c9986f98d4c8eef1e54df0e7b,https://computomics.com/home.html,computomics.com,computomics.com,4,1,completed,2024-07-14 22:44:14.985453,2024-07-15 00:34:36.558563,1fecc3a96c8cca2c244e40947664895f,0.560571,True,True


In [1]:
from sklearn.feature_extraction.text import CountVectorizer

In [2]:
vectorizer = CountVectorizer(stop_words="english", lowercase=True)

In [4]:
text = "This is some text but it has important terms like 'food' and 'drinks'."

In [11]:
vectorizer = vectorizer.fit([text])

In [13]:
vectorizer.get_feature_names_out().tolist()

['drinks', 'food', 'important', 'like', 'terms', 'text']

In [14]:
query = "food and drinks"

vectorizer = CountVectorizer(stop_words="english")

vectorizer.fit([query])
query_terms = [
    query,
    *vectorizer.get_feature_names_out()
]

In [2]:
from retriever_v2.utils import EMBEDDINGS_DIR
import numpy as np

In [4]:
embeddings = np.load(EMBEDDINGS_DIR / "all-mpnet-base-v2.npy")
ids = np.load(EMBEDDINGS_DIR / "ids.npy")

In [5]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer

In [8]:
vectorizer = CountVectorizer(stop_words="english")

In [9]:
query = "Best pizza and beer in Tübingen"

vectorizer.fit([query])
query_terms = list(
    {
        " ".join(query.split()),
        *vectorizer.get_feature_names_out()
    }
)
query_terms

['pizza', 'Best pizza and beer in Tübingen', 'best', 'beer', 'tübingen']

In [10]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-mpnet-base-v2')
query_embeddings = model.encode(query_terms)

C:\Users\Work\Desktop\Python projects\mse-group-project\venv\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [12]:
import pandas as pd
sim_df = pd.DataFrame(
    cosine_similarity(embeddings, query_embeddings),
    index=ids,
    columns=query_terms
)

In [17]:
sim_df = (
    sim_df
    .groupby(sim_df.index)
    .apply(lambda x: x.max(axis=0).sum() / len(query_terms))
)

In [20]:
for doc_id, score in sim_df.items():
    print(doc_id, type(score))
    break

00015cf9deb96f8073aac331866a4a19 <class 'float'>


In [ ]:
sim_df = (
    sim_df
    .groupby(sim_df.index)
    .apply(lambda x: x.max(axis=0).sum() / len(query_terms))
)
return [
    RetrievalScore(
        doc_id=str(doc_id),
        score=score,
        ranker="sim"
    )
    for doc_id, score in sim_df.items()
]